<a href="https://colab.research.google.com/github/kp-algo/encoder-decoder/blob/main/encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#packages
import numpy as np
import urllib.request, zipfile
import torch
import torch.nn as nn
import torch.nn.functional as f

In [2]:
#dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-11-02 16:33:55--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-11-02 16:33:55 (19.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
with open('input.txt', 'r', encoding='utf-8') as fh:
    text = fh.read()

chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

bos_id = stoi['\n']                          # the decoder's start token (newline, as in Karpathy's generate)

In [4]:
#hyperparams
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 192
block_size = 128
dropout = 0.2
head_size = 48
n_head = 4
n_layers = 3
vocab_size = len(chars)
batch_size = 32
learning_rate = 3e-4
src_len = 128            # characters the encoder reads
tgt_len = 128
max_iters = 5000
eval_interval = 500
eval_iters = 100

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

# Data Split



*   BART: requries two different inputs stream(one for encoder and one for decoder)
*   GPT: requires a single input stream(for decoder alone)



In [6]:
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - src_len - tgt_len - 1, (batch_size,))
    src = torch.stack([d[i : i + src_len] for i in ix])
    tgt = torch.stack([d[i + src_len : i + src_len + tgt_len] for i in ix])
    bos = torch.full((batch_size, 1), bos_id, dtype=torch.long)
    dec_in = torch.cat([bos, tgt[:, :-1]], dim=1)
    return src.to(device), dec_in.to(device), tgt.to(device)

In [7]:
def get_batch_gpt(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

#Architecture


*   Head: single attention head; fundamental unit of the architecture
*   MultiHead: stacks the multiple heads
*   FeedForward: simple linear neural network
*   Encoder Block: encoder; no masking
*   Decoder Block: decoder; masking
*   GPT: encoder-decoder; final model







In [8]:
class Head(nn.Module):
    def __init__(self, head_size, causal):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.causal = causal

    def forward(self, x, context=None):
        B,T,C = x.shape

        q = self.query(x)

        cxt = x if context is None else context

        k = self.key(cxt)
        v = self.value(cxt)

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        if self.causal:
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        wei = f.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out


In [9]:
class MultiHead(nn.Module):
    def __init__(self, head_size, num_heads, causal):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, causal) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, context=None):
        out = torch.cat([head(x, context) for head in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)

        return out

In [10]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd * 4),
            nn.ReLU(),
            nn.Linear(n_embd * 4, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        out = self.net(x)
        return out

In [11]:
class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd//n_head
        self.sa_head = MultiHead(head_size, n_head, causal=False)
        self.fwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa_head(self.ln1(x))
        x = x + self.fwd(self.ln2(x))
        return x

In [12]:
class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd//n_head
        self.sa_head = MultiHead(head_size, n_head, causal = True)
        self.ca_head = MultiHead(head_size, n_head, causal = False)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)
        self.fwd = FeedForward(n_embd)

    def forward(self, x, enc):
        x = x + self.sa_head(self.ln1(x))
        x = x + self.ca_head(self.ln2(x), enc)
        x = x + self.fwd(self.ln3(x))
        return x

#BART:


---


*   an autoregressive model with both **encoder** and **decoder** block connected with cross attention.
*   requires two separate inputs; one for encoder and other one for decoder.



In [13]:
class BART(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.pos_embd = nn.Embedding(block_size, n_embd)

        self.enc_blocks = nn.Sequential(*[EncoderBlock(n_embd, n_head) for _ in range(n_layers)])
        self.dec_blocks = nn.ModuleList([DecoderBlock(n_embd, n_head) for _ in range(n_layers)])

        self.ln_enc = nn.LayerNorm(n_embd)
        self.lnf = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def embed(self, ids):
        T = ids.shape[1]
        return self.token_embd(ids) + self.pos_embd(torch.arange(T, device=ids.device))

    def encode(self, src):
        return self.ln_enc(self.enc_blocks(self.embed(src)))

    def decode(self, dec_in, enc):
        x = self.embed(dec_in)
        for dec in self.dec_blocks:
            x = dec(x, enc)
        return self.lm_head(self.lnf(x))

    def forward(self, src, dec_in, targets=None):
        logits = self.decode(dec_in, self.encode(src))
        loss = None if targets is None else f.cross_entropy(
            logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    #claude generated
    @torch.no_grad()
    def generate(self, src, bos_id, max_new_tokens=64, temperature=0.8, top_k=40):
        self.eval()                                    # turns dropout off
        enc = self.encode(src)                         # (B, S, n_embd), computed once
        out = torch.full((src.shape[0], 1), bos_id, dtype=torch.long, device=src.device)

        for _ in range(max_new_tokens):
            dec_in = out[:, -block_size:]              # crop to the position table size
            logits = self.decode(dec_in, enc)[:, -1, :] / temperature     # last position only

            if top_k is not None:                      # keep only the k most likely tokens
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = f.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            out = torch.cat([out, next_tok], dim=1)

        return out[:, 1:]                              # drop the start token

In [14]:
m = BART().to(device)                         # your model class name
print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters')

src, dec_in, tgt = get_batch('train')
logits, loss = m(src, dec_in, tgt)
print(logits.shape, loss.item())             # (32, 128, 65), loss near ln(65) ≈ 4.17

3.160193 M parameters
torch.Size([32, 128, 65]) 4.309571266174316


#GPT:


---
*   an autoregressive model with n_layers of **decoder** blocks  
*   requires a single input stream for decoder


In [15]:
class GPT(nn.Module):
    def __init__(self, n_embd, n_head, n_layers):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.pos_embd = nn.Embedding(block_size, n_embd)

        head_size = n_embd // n_head
        self.sa  = nn.ModuleList([MultiHead(head_size, n_head, causal=True) for _ in range(n_layers)])
        self.fwd = nn.ModuleList([FeedForward(n_embd) for _ in range(n_layers)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(n_embd) for _ in range(n_layers)])

        self.lnf = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embd(idx) + self.pos_embd(torch.arange(T, device=idx.device))

        for sa, fwd, ln1, ln2 in zip(self.sa, self.fwd, self.ln1, self.ln2):
            x = x + sa(ln1(x))
            x = x + fwd(ln2(x))

        logits = self.lm_head(self.lnf(x))          # (B, T, vocab_size)
        loss = None if targets is None else f.cross_entropy(
            logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = f.softmax(logits[:, -1, :] / temperature, dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx

In [16]:
gpt = GPT(n_embd, n_head, n_layers).to(device)
print(sum(p.numel() for p in gpt.parameters()) / 1e6, 'M parameters')

src, dec_in, tgt = get_batch('train')
logits, loss = gpt(src, tgt)
print(logits.shape, loss.item())

1.382849 M parameters
torch.Size([32, 128, 65]) 4.299884796142578


#Output:


---



*   BART(with continuation)
*   GPT




In [17]:
@torch.no_grad()
def estimate_loss():
    m.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            s, d_in, t = get_batch(split)
            _, loss = m(s, d_in, t)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    m.train()
    return out

optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        L = estimate_loss()
        print(f"step {it}: train {L['train']:.3f}, val {L['val']:.3f}")

    s, d_in, t = get_batch('train')
    _, loss = m(s, d_in, t)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train 4.306, val 4.303
step 500: train 2.304, val 2.322
step 1000: train 2.007, val 2.078
step 1500: train 1.854, val 1.956
step 2000: train 1.734, val 1.876
step 2500: train 1.658, val 1.815
step 3000: train 1.605, val 1.757
step 3500: train 1.559, val 1.731
step 4000: train 1.533, val 1.716
step 4500: train 1.504, val 1.692
step 4999: train 1.482, val 1.675


In [20]:
src, _, tgt = get_batch('val')
gen = m.generate(src[:1], bos_id, max_new_tokens=200)

print('SOURCE:\n', decode(src[0].tolist()))
print('\nACTUAL:\n', decode(tgt[0].tolist()))
print('\nMODEL:\n', decode(gen[0].tolist()))

SOURCE:
 bedience;
Too little payment for so great a debt.
Such duty as the subject owes the prince
Even such a woman oweth to her husban

ACTUAL:
 d;
And when she is froward, peevish, sullen, sour,
And not obedient to his honest will,
What is she but a foul contending rebel


MODEL:
 ing to cannot
So the as would of me to a constitune
A for time myself.

RIVERS:
He sileman: not speak.

ISABELLA:
Yet's death, I conder, noble good the sword:
Henry your let as for not my conjusticati


In [18]:
@torch.no_grad()
def estimate_loss_gpt(model):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch_gpt(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out
optimizer = torch.optim.AdamW(gpt.parameters(), lr=learning_rate)

for it in range(max_iters):
    x, y = get_batch_gpt('train')
    _, loss = gpt(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
    optimizer.step()

    if it % eval_interval == 0 or it == max_iters - 1:
        L = estimate_loss_gpt(gpt)
        print(f"step {it}: train {L['train']:.3f}, val {L['val']:.3f}")

step 0: train 4.106, val 4.112
step 500: train 2.291, val 2.316
step 1000: train 2.007, val 2.069
step 1500: train 1.837, val 1.947
step 2000: train 1.727, val 1.870
step 2500: train 1.653, val 1.812
step 3000: train 1.607, val 1.769
step 3500: train 1.563, val 1.743
step 4000: train 1.535, val 1.716
step 4500: train 1.502, val 1.695
step 4999: train 1.486, val 1.672


In [19]:
src, _, tgt = get_batch('val')
gen = m.generate(src[:1], bos_id, max_new_tokens=200)

print('SOURCE:\n', decode(src[0].tolist()))
print('\nACTUAL:\n', decode(tgt[0].tolist()))
print('\nMODEL:\n', decode(gen[0].tolist()))

SOURCE:
 we might
beguile the old pantaloon.

HORTENSIO:
Madam, my instrument's in tune.

BIANCA:
Let's hear. O fie! the treble jars.

LU

ACTUAL:
 CENTIO:
Spit in the hole, man, and tune again.

BIANCA:
Now let me see if I can construe it: 'Hic ibat
Simois,' I know you not, 

MODEL:
 CAPULET:
I will the warrike and Lequers deserved,
To be my such of here or God, he to me my findly.
O charge anclent to this before, and not me as hold
The good out stands both broke of my lord.

That
